# Export deduplicated S3 clips for every Johnnie Walker stoppage

For every row in `johnnie_walker_black_label_events_2026-07-23.csv`, this notebook creates one video covering exactly:

- 10 seconds before `Event Time`
- 5 seconds after `Event Time`

Nearby S3 transport streams are stitched in recovered frame-timestamp order. Overlapping frames are removed using the existing 32-bit PTS recovery and covered-timeline logic, so duplicated footage is not written twice.

## 1. Configuration

In [16]:
from datetime import datetime, timedelta
from pathlib import Path
import hashlib
import re
import sys

import av
import boto3
import cv2
import numpy as np
import pandas as pd

working_directory = Path.cwd().resolve()
repository_candidates = []
for candidate in [working_directory, *working_directory.parents]:
    repository_candidates.extend([candidate, candidate / "24H_Insights"])
REPO_ROOT = next(
    (candidate for candidate in repository_candidates if (candidate / "VideoModule").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the 24H_Insights project root")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

AWS_PROFILE = "DashcamGlbDiageoProdDataContrib-522196013725"
S3_BUCKET = "diageo-prod-global-dashcam-mc-nuc-video"
S3_PREFIX = "cortexvpu-01a-005-41884872/"
VIDEO_EXTENSIONS = (".ts",)
CLIP_TIMESTAMP_PATTERN = re.compile(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}_\d+)")

PIPELINE_ROOT = REPO_ROOT / "stoppage_detection_and_classification"
PLC_EVENTS_DIR = PIPELINE_ROOT / "plc_stoppage_events"

EVENTS_CSV = (
    PLC_EVENTS_DIR
    / "output"
    / "01_prepare_events"
    / "johnnie_walker_black_label_events_2026-07-23.csv"
)
EVENT_TIMEZONE = "Europe/London"
SECONDS_BEFORE = 10.0
SECONDS_AFTER = 5.0
CANDIDATE_PADDING_SECONDS = 120.0

OUTPUT_DIR = PLC_EVENTS_DIR / "output" / "02_deduplicate_and_generate_event_clips"
SOURCE_CLIPS_DIR = OUTPUT_DIR / "source_s3_clips"
STITCHED_CLIPS_DIR = OUTPUT_DIR / "stitched_event_clips"
MANIFEST_PATH = OUTPUT_DIR / "stoppage_clip_manifest.csv"
TIMELINE_CACHE_PATH = OUTPUT_DIR / "source_timeline_summary.csv"

# Keep False until the profile and queue have been reviewed.
RUN_EXPORT = True
MAX_EVENTS = None  # Use 3 for a first test; None processes every CSV row.
REPROCESS_EXISTING = False
REPROCESS_SOURCE_TIMELINES = False

UINT32_MODULUS = 2**32
EPOCH = datetime(1970, 1, 1)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_CLIPS_DIR.mkdir(parents=True, exist_ok=True)
STITCHED_CLIPS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Events:  {EVENTS_CSV}")
print(f"Output:  {STITCHED_CLIPS_DIR}")
print(f"Window:  -{SECONDS_BEFORE:g}s / +{SECONDS_AFTER:g}s")
print(f"Run:     {RUN_EXPORT}")

Events:  C:\Users\TomKitching\Documents\24H_INSIGHTS\24H_Insights\classifier\plc_stoppage_events\output\01_prepare_events\johnnie_walker_black_label_events_2026-07-23.csv
Output:  C:\Users\TomKitching\Documents\24H_INSIGHTS\24H_Insights\classifier\plc_stoppage_events\output\02_deduplicate_and_generate_event_clips\stitched_event_clips
Window:  -10s / +5s
Run:     True


## 2. Load all stoppages and list nearby S3 videos

In [17]:
events = pd.read_csv(EVENTS_CSV)
if "Event Time" not in events.columns:
    raise ValueError("The event CSV does not contain an 'Event Time' column")
if events.empty:
    raise ValueError("The event CSV is empty")

# Every CSV row is retained, as requested.
local_event_times = pd.to_datetime(events["Event Time"], errors="raise").dt.tz_localize(
    EVENT_TIMEZONE,
    ambiguous="raise",
    nonexistent="raise",
)
events = events.reset_index(drop=True)
events.insert(0, "event_id", [f"stoppage_{index + 1:06d}" for index in range(len(events))])
events["event_time_local"] = local_event_times.reset_index(drop=True)
events["event_time_utc"] = local_event_times.dt.tz_convert("UTC").dt.tz_localize(None).reset_index(drop=True)
events["window_start_utc"] = events["event_time_utc"] - pd.to_timedelta(SECONDS_BEFORE, unit="s")
events["window_end_utc"] = events["event_time_utc"] + pd.to_timedelta(SECONDS_AFTER, unit="s")

listing_start = events["window_start_utc"].min() - pd.Timedelta(seconds=CANDIDATE_PADDING_SECONDS)
listing_end = events["window_end_utc"].max() + pd.Timedelta(seconds=CANDIDATE_PADDING_SECONDS)

session = boto3.Session(profile_name=AWS_PROFILE)
s3_client = session.client("s3")
video_rows = []
paginator = s3_client.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX):
    for s3_object in page.get("Contents", []):
        s3_key = str(s3_object["Key"])
        if not s3_key.lower().endswith(VIDEO_EXTENSIONS):
            continue
        timestamp_match = CLIP_TIMESTAMP_PATTERN.search(Path(s3_key).name)
        if timestamp_match is None:
            continue
        filename_timestamp = pd.to_datetime(
            timestamp_match.group(1),
            format="%Y-%m-%d_%H-%M-%S_%f",
        )
        if listing_start <= filename_timestamp <= listing_end:
            video_rows.append({
                "s3_key": s3_key,
                "filename_timestamp": filename_timestamp,
                "size_bytes": int(s3_object.get("Size", 0)),
            })

s3_videos = pd.DataFrame(video_rows).sort_values(["filename_timestamp", "s3_key"]).reset_index(drop=True)
if s3_videos.empty:
    raise RuntimeError("No S3 videos were found around the event date range")

print(f"Stoppages loaded: {len(events):,}")
print(f"S3 videos listed: {len(s3_videos):,}")
print(f"Listing range:    {listing_start} to {listing_end} UTC")
display(events[["event_id", "Event Time", "window_start_utc", "window_end_utc"]].head())

Stoppages loaded: 251
S3 videos listed: 15,339
Listing range:    2026-07-01 00:40:17.202000 to 2026-07-08 07:42:07.052000 UTC


,event_id,Event Time,window_start_utc,window_end_utc
0,stoppage_000001,2026-07-08 08:40:02.052,2026-07-08 07:39:52.052,2026-07-08 07:40:07.052
1,stoppage_000002,2026-07-07 15:39:09.325,2026-07-07 14:38:59.325,2026-07-07 14:39:14.325
2,stoppage_000003,2026-07-07 15:03:56.285,2026-07-07 14:03:46.285,2026-07-07 14:04:01.285
3,stoppage_000004,2026-07-07 14:48:29.160,2026-07-07 13:48:19.160,2026-07-07 13:48:34.160
4,stoppage_000005,2026-07-07 14:25:38.791,2026-07-07 13:25:28.791,2026-07-07 13:25:43.791


## 3. Select and download candidate source clips

Each stoppage uses videos whose filename timestamps are within two minutes of its event window, plus the immediately preceding video. Downloads are shared between stoppages.

In [18]:
candidate_keys_by_event = {}
all_candidate_keys = set()
video_timestamps = s3_videos["filename_timestamp"].tolist()

for event in events.itertuples(index=False):
    padded_start = event.window_start_utc - pd.Timedelta(seconds=CANDIDATE_PADDING_SECONDS)
    padded_end = event.window_end_utc + pd.Timedelta(seconds=CANDIDATE_PADDING_SECONDS)
    matching_indices = [
        index
        for index, timestamp in enumerate(video_timestamps)
        if padded_start <= timestamp <= padded_end
    ]
    if matching_indices and min(matching_indices) > 0:
        matching_indices.insert(0, min(matching_indices) - 1)
    event_keys = s3_videos.iloc[sorted(set(matching_indices))]["s3_key"].tolist()
    candidate_keys_by_event[event.event_id] = event_keys
    all_candidate_keys.update(event_keys)

selected_sources = s3_videos[s3_videos["s3_key"].isin(all_candidate_keys)].copy()
selected_sources["local_path"] = selected_sources["s3_key"].map(
    lambda key: str(SOURCE_CLIPS_DIR / Path(key).name)
)

print(f"Unique source clips required: {len(selected_sources):,}")
if RUN_EXPORT:
    reused_count = 0
    downloaded_count = 0
    for source in selected_sources.itertuples(index=False):
        local_path = Path(source.local_path)
        if local_path.is_file() and local_path.stat().st_size > 0:
            reused_count += 1
            if local_path.stat().st_size != source.size_bytes:
                print(
                    f"WARNING: reusing {local_path.name} despite size mismatch "
                    f"(local={local_path.stat().st_size:,}, S3={source.size_bytes:,})"
                )
            continue
        partial_path = local_path.with_suffix(local_path.suffix + ".part")
        s3_client.download_file(S3_BUCKET, source.s3_key, str(partial_path))
        partial_path.replace(local_path)
        downloaded_count += 1
    print(f"Reused existing source clips: {reused_count:,}")
    print(f"Downloaded source clips:      {downloaded_count:,}")
else:
    print("Downloads disabled. Set RUN_EXPORT=True after reviewing the counts.")

local_path_by_key = {
    row.s3_key: Path(row.local_path)
    for row in selected_sources.itertuples(index=False)
}
display(selected_sources.head())

Unique source clips required: 1,754
Reused existing source clips: 1,754
Downloaded source clips:      0


,s3_key,filename_timestamp,size_bytes,local_path
0,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,2026-07-01 00:40:28.133333,26072780,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...
1,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,2026-07-01 00:40:40.133333,17091644,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...
2,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,2026-07-01 00:40:47.133333,9917940,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...
3,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,2026-07-01 00:40:50.133333,9963812,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...
4,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,2026-07-01 00:40:53.133333,69368240,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...


## 4. Recover exact frame timelines

This is the original notebook's 32-bit PTS recovery logic. It gives each decoded frame an absolute UTC presentation timestamp.

In [19]:
def iter_timestamped_frames(video_path, filename_time_estimate, include_pixels=False):
    previous_recovered_timestamp = None
    with av.open(str(video_path)) as container:
        for frame in container.decode(video=0):
            if frame.pts is None or frame.time_base is None:
                continue
            ticks_per_second = round(1 / float(frame.time_base))
            pts_32bit = int(frame.pts) % UINT32_MODULUS
            time_estimate = previous_recovered_timestamp or filename_time_estimate
            estimated_ticks = (time_estimate - EPOCH).total_seconds() * ticks_per_second
            nearest_rollover = round((estimated_ticks - pts_32bit) / UINT32_MODULUS)
            full_pts = pts_32bit + nearest_rollover * UINT32_MODULUS
            recovered_timestamp = EPOCH + timedelta(seconds=full_pts / ticks_per_second)
            previous_recovered_timestamp = recovered_timestamp
            pixels = frame.to_ndarray(format="bgr24") if include_pixels else None
            yield pd.Timestamp(recovered_timestamp), pixels

timeline_columns = [
    "s3_key", "filename_timestamp", "local_path", "recovered_start",
    "recovered_end", "decoded_frame_count", "median_frame_duration_seconds",
]
cached_timelines = pd.DataFrame(columns=timeline_columns)
if TIMELINE_CACHE_PATH.is_file() and not REPROCESS_SOURCE_TIMELINES:
    cached_timelines = pd.read_csv(TIMELINE_CACHE_PATH)
    for column in ["filename_timestamp", "recovered_start", "recovered_end"]:
        cached_timelines[column] = pd.to_datetime(cached_timelines[column])
    cached_timelines = cached_timelines[
        cached_timelines["s3_key"].isin(selected_sources["s3_key"])
    ].copy()

cached_keys = set(cached_timelines["s3_key"])
sources_to_process = selected_sources[
    ~selected_sources["s3_key"].isin(cached_keys)
]
timeline_rows = []
if RUN_EXPORT:
    for source in sources_to_process.itertuples(index=False):
        local_path = Path(source.local_path)
        timestamps = [
            timestamp
            for timestamp, _ in iter_timestamped_frames(
                local_path,
                source.filename_timestamp.to_pydatetime(),
                include_pixels=False,
            )
        ]
        if not timestamps:
            continue
        unique_timestamps = sorted(set(timestamps))
        differences = pd.Series(unique_timestamps).diff().dt.total_seconds()
        positive_differences = differences[differences > 0]
        timeline_rows.append({
            "s3_key": source.s3_key,
            "filename_timestamp": source.filename_timestamp,
            "local_path": str(local_path),
            "recovered_start": unique_timestamps[0],
            "recovered_end": unique_timestamps[-1],
            "decoded_frame_count": len(unique_timestamps),
            "median_frame_duration_seconds": float(positive_differences.median()),
        })

new_timelines = pd.DataFrame(timeline_rows, columns=timeline_columns)
timeline_summary = pd.concat([cached_timelines, new_timelines], ignore_index=True)
if not new_timelines.empty or (not TIMELINE_CACHE_PATH.exists() and not timeline_summary.empty):
    timeline_summary.to_csv(TIMELINE_CACHE_PATH, index=False)
if RUN_EXPORT and timeline_summary.empty:
    raise RuntimeError("No usable source timelines were recovered")
if not timeline_summary.empty:
    timeline_summary = timeline_summary.sort_values(["recovered_start", "s3_key"]).reset_index(drop=True)
    print(f"Loaded cached source timelines: {len(cached_timelines):,}")
    print(f"Processed new source timelines: {len(new_timelines):,}")
    print(f"Timeline cache: {TIMELINE_CACHE_PATH}")
    display(timeline_summary.head())

Loaded cached source timelines: 0
Processed new source timelines: 1,754
Timeline cache: C:\Users\TomKitching\Documents\24H_INSIGHTS\24H_Insights\classifier\plc_stoppage_events\output\02_deduplicate_and_generate_event_clips\source_timeline_summary.csv


,s3_key,filename_timestamp,local_path,recovered_start,recovered_end,decoded_frame_count,median_frame_duration_seconds
0,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,2026-07-01 00:40:28.133333,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,2026-07-01 00:40:25.133333,2026-07-01 00:40:39.550000,866,0.016667
1,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,2026-07-01 00:40:40.133333,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,2026-07-01 00:40:37.133333,2026-07-01 00:40:46.550000,566,0.016667
2,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,2026-07-01 00:40:47.133333,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,2026-07-01 00:40:44.133333,2026-07-01 00:40:49.566667,327,0.016667
3,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,2026-07-01 00:40:50.133333,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,2026-07-01 00:40:47.133333,2026-07-01 00:40:52.566667,327,0.016667
4,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,2026-07-01 00:40:53.133333,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,2026-07-01 00:40:50.133333,2026-07-01 00:41:28.600000,2309,0.016667


## 5. Write one deduplicated clip per stoppage

Sources are decoded in recovered-time order. A frame is written only when it lies inside the event window and is later than the covered timeline by half a frame duration. This removes duplicated overlap from stitched videos.

In [20]:
def safe_event_filename(event_id, event_time_utc):
    timestamp_text = pd.Timestamp(event_time_utc).strftime("%Y-%m-%d_%H-%M-%S_%f")
    return f"{event_id}__{timestamp_text}__10s_before_5s_after.mp4"

def write_event_clip(event, source_timelines, output_path):
    overlapping = source_timelines[
        (source_timelines["recovered_start"] <= event.window_end_utc)
        & (source_timelines["recovered_end"] >= event.window_start_utc)
    ].sort_values(["recovered_start", "s3_key"])
    if overlapping.empty:
        return {"status": "no_footage", "frames_written": 0, "source_keys": []}

    frame_durations = overlapping["median_frame_duration_seconds"].dropna()
    frame_duration = float(frame_durations.median()) if not frame_durations.empty else 1 / 30
    output_fps = 1.0 / frame_duration if frame_duration > 0 else 30.0
    frame_tolerance = pd.Timedelta(seconds=frame_duration / 2)
    writer = None
    covered_timeline_end = None
    frames_written = 0
    first_written_timestamp = None
    last_written_timestamp = None
    used_source_keys = []

    try:
        for source in overlapping.itertuples(index=False):
            source_wrote_frame = False
            for timestamp, frame in iter_timestamped_frames(
                source.local_path,
                source.filename_timestamp.to_pydatetime(),
                include_pixels=True,
            ):
                if timestamp < event.window_start_utc:
                    continue
                if timestamp > event.window_end_utc:
                    break
                # The later source may repeat footage already written by an earlier source.
                if covered_timeline_end is not None and timestamp <= covered_timeline_end + frame_tolerance:
                    continue
                if writer is None:
                    height, width = frame.shape[:2]
                    writer = cv2.VideoWriter(
                        str(output_path),
                        cv2.VideoWriter_fourcc(*"mp4v"),
                        output_fps,
                        (width, height),
                    )
                    if not writer.isOpened():
                        raise RuntimeError(f"Could not open video writer: {output_path}")
                writer.write(frame)
                frames_written += 1
                source_wrote_frame = True
                first_written_timestamp = first_written_timestamp or timestamp
                last_written_timestamp = timestamp
                covered_timeline_end = timestamp
            if source_wrote_frame:
                used_source_keys.append(source.s3_key)
    finally:
        if writer is not None:
            writer.release()

    status = "ok" if frames_written else "no_frames_in_window"
    return {
        "status": status,
        "frames_written": frames_written,
        "source_keys": used_source_keys,
        "first_frame_utc": first_written_timestamp,
        "last_frame_utc": last_written_timestamp,
        "output_fps": output_fps,
    }

manifest_rows = []
if not RUN_EXPORT:
    print("Export disabled. Set RUN_EXPORT=True and rerun from Section 3.")
else:
    requested_events = events.head(MAX_EVENTS) if MAX_EVENTS is not None else events
    skip_chapter_5 = False
    if MANIFEST_PATH.is_file() and not REPROCESS_EXISTING:
        cached_manifest = pd.read_csv(MANIFEST_PATH).drop_duplicates("event_id", keep="last")
        cached_manifest["event_time_utc"] = pd.to_datetime(cached_manifest["event_time_utc"])
        expected_event_times = requested_events.set_index("event_id")["event_time_utc"]
        cached_event_times = cached_manifest.set_index("event_id")["event_time_utc"]
        same_events = set(cached_event_times.index) == set(expected_event_times.index)
        same_times = same_events and cached_event_times.sort_index().equals(
            expected_event_times.sort_index()
        )
        completed_without_video = {"no_footage", "no_frames_in_window"}
        outputs_still_valid = all(
            str(row.status) in completed_without_video
            or (
                (STITCHED_CLIPS_DIR / safe_event_filename(row.event_id, row.event_time_utc)).is_file()
                and (STITCHED_CLIPS_DIR / safe_event_filename(row.event_id, row.event_time_utc)).stat().st_size > 0
            )
            for row in cached_manifest.itertuples(index=False)
        )
        skip_chapter_5 = same_times and outputs_still_valid
        if skip_chapter_5:
            manifest_rows = cached_manifest.to_dict("records")
            print("Chapter 5 unchanged: loaded the existing manifest and skipped stitching.")
    reused_stitched_count = 0
    newly_stitched_count = 0
    events_to_process = requested_events.head(0) if skip_chapter_5 else requested_events
    for event_number, event in enumerate(events_to_process.itertuples(index=False), start=1):
        output_path = STITCHED_CLIPS_DIR / safe_event_filename(event.event_id, event.event_time_utc)
        if output_path.is_file() and output_path.stat().st_size > 0 and not REPROCESS_EXISTING:
            result = {"status": "already_exists", "frames_written": "", "source_keys": []}
            reused_stitched_count += 1
        else:
            result = write_event_clip(event, timeline_summary, output_path)
            if result["status"] == "ok":
                newly_stitched_count += 1
        manifest_rows.append({
            "event_id": event.event_id,
            "event_time_local": event.event_time_local,
            "event_time_utc": event.event_time_utc,
            "window_start_utc": event.window_start_utc,
            "window_end_utc": event.window_end_utc,
            "status": result["status"],
            "frames_written": result.get("frames_written", ""),
            "first_frame_utc": result.get("first_frame_utc", ""),
            "last_frame_utc": result.get("last_frame_utc", ""),
            "source_clip_count": len(result.get("source_keys", [])),
            "source_s3_keys": "|".join(result.get("source_keys", [])),
            "output_video": str(output_path) if output_path.exists() else "",
        })
        print(f"[{event_number}/{len(events_to_process)}] {event.event_id}: {result['status']}")

    print(f"Reused existing stitched clips: {reused_stitched_count:,}")
    print(f"Newly stitched clips:          {newly_stitched_count:,}")
    manifest = pd.DataFrame(manifest_rows)
    manifest.to_csv(MANIFEST_PATH, index=False)
    print()
    print(manifest["status"].value_counts().to_string())
    print(f"Clips:    {STITCHED_CLIPS_DIR}")
    print(f"Manifest: {MANIFEST_PATH}")

Chapter 5 unchanged: loaded the existing manifest and skipped stitching.
Reused existing stitched clips: 0
Newly stitched clips:          0

status
no_footage    153
ok             98
Clips:    C:\Users\TomKitching\Documents\24H_INSIGHTS\24H_Insights\classifier\plc_stoppage_events\output\02_deduplicate_and_generate_event_clips\stitched_event_clips
Manifest: C:\Users\TomKitching\Documents\24H_INSIGHTS\24H_Insights\classifier\plc_stoppage_events\output\02_deduplicate_and_generate_event_clips\stoppage_clip_manifest.csv


In [21]:
# Save the already-computed Chapter 4 output, then reconstruct frame provenance.
timeline_cache_path = globals().get(
    "TIMELINE_CACHE_PATH", OUTPUT_DIR / "source_timeline_summary.csv"
)
if not timeline_cache_path.is_file():
    timeline_summary.to_csv(timeline_cache_path, index=False)
    print(f"Saved Chapter 4 timeline cache: {timeline_cache_path}")

clip_name = "stoppage_000083__2026-07-06_06-50-28_588000__10s_before_5s_after.mp4"
video_path = STITCHED_CLIPS_DIR / clip_name
event_time_utc = pd.to_datetime(
    clip_name.split("__")[1], format="%Y-%m-%d_%H-%M-%S_%f"
)
window_start_utc = event_time_utc - pd.Timedelta(seconds=SECONDS_BEFORE)
window_end_utc = event_time_utc + pd.Timedelta(seconds=SECONDS_AFTER)

overlapping = timeline_summary[
    (timeline_summary["recovered_start"] <= window_end_utc)
    & (timeline_summary["recovered_end"] >= window_start_utc)
].sort_values(["recovered_start", "s3_key"])
if overlapping.empty:
    raise RuntimeError("No downloaded source clips overlap this event window")

frame_durations = overlapping["median_frame_duration_seconds"].dropna()
frame_duration = float(frame_durations.median()) if not frame_durations.empty else 1 / 30
frame_tolerance = pd.Timedelta(seconds=frame_duration / 2)
covered_timeline_end = None
frame_rows = []
for source in overlapping.itertuples(index=False):
    for source_frame_index, (timestamp, _) in enumerate(iter_timestamped_frames(
        source.local_path,
        source.filename_timestamp.to_pydatetime(),
        include_pixels=False,
    )):
        if timestamp < window_start_utc:
            continue
        if timestamp > window_end_utc:
            break
        if covered_timeline_end is not None and timestamp <= covered_timeline_end + frame_tolerance:
            continue
        frame_rows.append({
            "stitched_video_name": video_path.name,
            "stitched_frame_index": len(frame_rows),
            "source_s3_key": source.s3_key,
            "source_video_name": Path(source.local_path).name,
            "source_video_path": str(Path(source.local_path).resolve()),
            "source_frame_index": source_frame_index,
            "frame_timestamp_utc": timestamp,
        })
        covered_timeline_end = timestamp

frame_utc_timestamps = pd.DataFrame(frame_rows)
frame_utc_timestamps["time_delta_from_previous_frame_seconds"] = (
    frame_utc_timestamps["frame_timestamp_utc"].diff().dt.total_seconds()
)
frame_utc_timestamps["source_clip_changed"] = (
    frame_utc_timestamps["source_s3_key"].ne(
        frame_utc_timestamps["source_s3_key"].shift()
    )
)
frame_utc_timestamps["frame_timestamp_utc"] = (
    frame_utc_timestamps["frame_timestamp_utc"].dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
)
csv_path = video_path.with_name(f"{video_path.stem}_frame_utc_timestamps.csv")
frame_utc_timestamps.to_csv(csv_path, index=False)
print(f"Video:  {video_path}")
print(f"Frames: {len(frame_utc_timestamps):,}")
print(f"Source clips: {frame_utc_timestamps['source_s3_key'].nunique():,}")
print(f"CSV:    {csv_path}")
display(frame_utc_timestamps)
display(frame_utc_timestamps[frame_utc_timestamps["source_clip_changed"]])

Video:  C:\Users\TomKitching\Documents\24H_INSIGHTS\24H_Insights\classifier\plc_stoppage_events\output\02_deduplicate_and_generate_event_clips\stitched_event_clips\stoppage_000083__2026-07-06_06-50-28_588000__10s_before_5s_after.mp4
Frames: 682
Source clips: 2
CSV:    C:\Users\TomKitching\Documents\24H_INSIGHTS\24H_Insights\classifier\plc_stoppage_events\output\02_deduplicate_and_generate_event_clips\stitched_event_clips\stoppage_000083__2026-07-06_06-50-28_588000__10s_before_5s_after_frame_utc_timestamps.csv


,stitched_video_name,stitched_frame_index,source_s3_key,source_video_name,source_video_path,source_frame_index,frame_timestamp_utc,time_delta_from_previous_frame_seconds,source_clip_changed
0,stoppage_000083__2026-07-06_06-50-28_588000__1...,0,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-06_06-50-23...,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,0,2026-07-06T06:50:20.850000Z,NaN,True
1,stoppage_000083__2026-07-06_06-50-28_588000__1...,1,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-06_06-50-23...,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,1,2026-07-06T06:50:20.866667Z,0.016667,False
2,stoppage_000083__2026-07-06_06-50-28_588000__1...,2,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-06_06-50-23...,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,2,2026-07-06T06:50:20.883333Z,0.016666,False
3,stoppage_000083__2026-07-06_06-50-28_588000__1...,3,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-06_06-50-23...,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,3,2026-07-06T06:50:20.900000Z,0.016667,False
4,stoppage_000083__2026-07-06_06-50-28_588000__1...,4,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-06_06-50-23...,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,4,2026-07-06T06:50:20.916667Z,0.016667,False
...,...,...,...,...,...,...,...,...,...
677,stoppage_000083__2026-07-06_06-50-28_588000__1...,677,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-06_06-50-30...,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,340,2026-07-06T06:50:33.516667Z,0.016667,False
678,stoppage_000083__2026-07-06_06-50-28_588000__1...,678,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-06_06-50-30...,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,341,2026-07-06T06:50:33.533333Z,0.016666,False
679,stoppage_000083__2026-07-06_06-50-28_588000__1...,679,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-06_06-50-30...,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,342,2026-07-06T06:50:33.550000Z,0.016667,False
680,stoppage_000083__2026-07-06_06-50-28_588000__1...,680,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-06_06-50-30...,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,343,2026-07-06T06:50:33.566667Z,0.016667,False


,stitched_video_name,stitched_frame_index,source_s3_key,source_video_name,source_video_path,source_frame_index,frame_timestamp_utc,time_delta_from_previous_frame_seconds,source_clip_changed
0,stoppage_000083__2026-07-06_06-50-28_588000__1...,0,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-06_06-50-23...,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,0,2026-07-06T06:50:20.850000Z,NaN,True
337,stoppage_000083__2026-07-06_06-50-28_588000__1...,337,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,cortexvpu-01a-005-41884872_2026-07-06_06-50-30...,C:\Users\TomKitching\Documents\24H_INSIGHTS\24...,0,2026-07-06T06:50:27.850000Z,1.4,True
